# Telco Customer Churn: Rewritten Pipeline

What changed from the first version:
1. **Thresholds are chosen on out-of-fold training predictions, never on the test set.** The test set is scored once.
2. **Every model gets the same treatment** (its own OOF-tuned threshold), so the comparison is fair.
3. **Bootstrap confidence intervals** show whether the model differences are real or noise.
4. `TotalCharges` blanks (tenure 0) are filled with 0, and tenure bins include tenure 0.
5. Risk bands are derived from the decision threshold instead of arbitrary cut-offs.
6. Added a gain/lift table and an expected-profit curve (cost inputs are **placeholders**, replace with real numbers).
7. Dead code and duplicate cells removed.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV, StratifiedKFold, cross_val_predict, train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=UserWarning)

RANDOM_STATE = 42
DATA_PATH = "Telco-Customer-Churn.csv"
rng = np.random.default_rng(RANDOM_STATE)

## 1. Load and clean

In [ ]:
churn = pd.read_csv(DATA_PATH)

churn["TotalCharges"] = pd.to_numeric(churn["TotalCharges"], errors="coerce")
blank_total = churn["TotalCharges"].isna()
print(f"Blank TotalCharges rows: {blank_total.sum()} "
      f"(all tenure == 0: {bool((churn.loc[blank_total, 'tenure'] == 0).all())})")
# New customers who haven't been billed yet: the correct value is 0, not the mean.
churn["TotalCharges"] = churn["TotalCharges"].fillna(0)

print("Duplicate customer IDs:", churn["customerID"].duplicated().sum())
print(churn["Churn"].value_counts(normalize=True).round(3).to_dict())

y = (churn["Churn"] == "Yes").astype(int)
X = churn.drop(columns=["Churn", "customerID"]).copy()
X["SeniorCitizen"] = X["SeniorCitizen"].map({0: "No", 1: "Yes"})  # binary flag, treat as categorical

## 2. Quick EDA

In [ ]:
eda = churn.copy()
eda["TenureGroup"] = pd.cut(
    eda["tenure"], bins=[-1, 12, 24, 36, 48, 60, 72],
    labels=["0-12", "13-24", "25-36", "37-48", "49-60", "61-72"],
)
for col in ["Contract", "TenureGroup", "InternetService", "PaymentMethod"]:
    summary = (
        eda.groupby(col, observed=True)
        .agg(Customers=("customerID", "count"),
             ChurnRate=("Churn", lambda s: (s == "Yes").mean()),
             AvgMonthly=("MonthlyCharges", "mean"))
        .sort_values("ChurnRate", ascending=False)
    )
    print(f"\n--- {col} ---")
    print(summary.round(3))

## 3. Split and preprocessing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

numerical_features = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_features = [c for c in X.columns if c not in numerical_features]

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scaler", StandardScaler())]), numerical_features),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("encoder", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
])

## 4. Models
The XGBoost search optimizes PR-AUC, which does not depend on a threshold. The threshold is chosen afterwards.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

logreg = Pipeline([("prep", preprocessor),
                   ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))])

forest = Pipeline([("prep", preprocessor),
                   ("clf", RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                                  min_samples_leaf=3, n_jobs=-1,
                                                  random_state=RANDOM_STATE))])

xgb_base = Pipeline([("prep", preprocessor),
                     ("clf", XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                                           subsample=0.8, colsample_bytree=0.8,
                                           eval_metric="logloss", random_state=RANDOM_STATE))])

param_distributions = {
    "clf__n_estimators": [200, 300, 500, 800],
    "clf__learning_rate": [0.01, 0.02, 0.03, 0.05, 0.1],
    "clf__max_depth": [2, 3, 4, 5],
    "clf__min_child_weight": [1, 3, 5],
    "clf__subsample": [0.7, 0.8, 1.0],
    "clf__colsample_bytree": [0.6, 0.8, 1.0],
    "clf__reg_alpha": [0, 0.1, 0.5],
    "clf__reg_lambda": [0.5, 1.0, 2.0],
    "clf__gamma": [0, 0.1, 0.3],
}
search = RandomizedSearchCV(
    xgb_base, param_distributions, n_iter=30, scoring="average_precision",
    cv=cv, random_state=RANDOM_STATE, n_jobs=-1,
)
search.fit(X_train, y_train)
print("Best CV PR-AUC:", round(search.best_score_, 4))
print("Best params:", search.best_params_)

models = {
    "Logistic Regression": logreg,
    "Random Forest": forest,
    "XGBoost (baseline)": xgb_base,
    "XGBoost (tuned)": search.best_estimator_,
}

## 5. Threshold selection on out-of-fold training predictions
Each model gets its own threshold, chosen **without touching the test set**.

In [ ]:
THRESHOLD_GRID = np.round(np.arange(0.10, 0.81, 0.01), 2)


def best_f1_threshold(y_true, proba):
    scores = [f1_score(y_true, proba >= t, zero_division=0) for t in THRESHOLD_GRID]
    return float(THRESHOLD_GRID[int(np.argmax(scores))])


oof_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE + 1)
oof_proba = {}
thresholds = {}
for name, model in models.items():
    oof_proba[name] = cross_val_predict(
        model, X_train, y_train, cv=oof_cv, method="predict_proba", n_jobs=-1
    )[:, 1]
    thresholds[name] = best_f1_threshold(y_train, oof_proba[name])
    print(f"{name:22s} OOF-selected threshold: {thresholds[name]:.2f}")

## 6. One-shot evaluation on the test set

In [ ]:
test_proba = {}
rows = {}
for name, model in models.items():
    fitted = model.fit(X_train, y_train)
    p = fitted.predict_proba(X_test)[:, 1]
    pred = (p >= thresholds[name]).astype(int)
    test_proba[name] = p
    rows[name] = {
        "Threshold": thresholds[name],
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, p),
        "PR-AUC": average_precision_score(y_test, p),
    }

model_compare = pd.DataFrame(rows).T.round(4)
print(model_compare.sort_values("PR-AUC", ascending=False))

## 7. Are the differences real? Bootstrap confidence intervals
With ~370 churners in the test set, small gaps are usually noise.

In [ ]:
def bootstrap_diff(name_a, name_b, n_boot=1000):
    y_arr = y_test.to_numpy()
    pa, pb = test_proba[name_a], test_proba[name_b]
    ta, tb = thresholds[name_a], thresholds[name_b]
    diffs = {"F1": [], "ROC-AUC": [], "PR-AUC": []}
    n = len(y_arr)
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        yy = y_arr[idx]
        if yy.min() == yy.max():
            continue
        diffs["F1"].append(f1_score(yy, pa[idx] >= ta, zero_division=0)
                           - f1_score(yy, pb[idx] >= tb, zero_division=0))
        diffs["ROC-AUC"].append(roc_auc_score(yy, pa[idx]) - roc_auc_score(yy, pb[idx]))
        diffs["PR-AUC"].append(average_precision_score(yy, pa[idx])
                               - average_precision_score(yy, pb[idx]))
    out = {m: (np.mean(v), *np.percentile(v, [2.5, 97.5])) for m, v in diffs.items()}
    return pd.DataFrame(out, index=["mean diff", "ci low", "ci high"]).T.round(4)


print("XGBoost (tuned) minus Logistic Regression:")
print(bootstrap_diff("XGBoost (tuned)", "Logistic Regression"))
print("\nIf the interval contains 0, the models are not distinguishable on this test set.")

## 8. Final model: confusion matrix, gain and lift

In [ ]:
FINAL_NAME = "XGBoost (tuned)"  # change this if the bootstrap above says a simpler model is just as good
final_model = models[FINAL_NAME]
final_threshold = thresholds[FINAL_NAME]
final_proba = test_proba[FINAL_NAME]
final_pred = (final_proba >= final_threshold).astype(int)

print(f"{FINAL_NAME} @ threshold {final_threshold:.2f}")
print(confusion_matrix(y_test, final_pred))

scored = pd.DataFrame({"y": y_test.to_numpy(), "p": final_proba}).sort_values("p", ascending=False)
scored["decile"] = pd.qcut(np.arange(len(scored)), 10, labels=range(1, 11))
lift = scored.groupby("decile", observed=True).agg(Customers=("y", "size"), Churners=("y", "sum"))
lift["CumChurnCaptured"] = lift["Churners"].cumsum() / lift["Churners"].sum()
lift["Lift"] = (lift["Churners"] / lift["Customers"]) / scored["y"].mean()
print(lift.round(3))

## 9. Expected-profit view (placeholder economics)
**Replace these three numbers with real ones.** They are illustrative, not from the data.

In [ ]:
VALUE_IF_SAVED = 500.0   # value of retaining a churner (placeholder)
OFFER_COST = 50.0        # cost of contacting/offering to one customer (placeholder)
SAVE_RATE = 0.30         # share of targeted churners the offer actually retains (placeholder)


def expected_profit(y_true, proba, t):
    targeted = proba >= t
    saved = (targeted & (y_true == 1)).sum() * SAVE_RATE * VALUE_IF_SAVED
    return saved - targeted.sum() * OFFER_COST


y_tr = y_train.to_numpy()
profit_curve = [expected_profit(y_tr, oof_proba[FINAL_NAME], t) for t in THRESHOLD_GRID]
profit_threshold = float(THRESHOLD_GRID[int(np.argmax(profit_curve))])
print(f"F1-optimal threshold: {final_threshold:.2f} | profit-optimal (placeholder costs): {profit_threshold:.2f}")

plt.figure(figsize=(7, 4))
plt.plot(THRESHOLD_GRID, profit_curve)
plt.axvline(final_threshold, ls="--", label="F1 threshold")
plt.axvline(profit_threshold, ls=":", label="profit threshold")
plt.xlabel("Threshold")
plt.ylabel("Expected profit on training OOF (placeholder units)")
plt.legend()
plt.tight_layout()
plt.show()

## 10. Explainability (SHAP)

In [ ]:
try:
    import shap

    fitted_prep = final_model.named_steps["prep"]
    fitted_clf = final_model.named_steps["clf"]
    X_test_t = fitted_prep.transform(X_test)
    if hasattr(X_test_t, "toarray"):
        X_test_t = X_test_t.toarray()
    names = fitted_prep.get_feature_names_out()

    shap_values = shap.TreeExplainer(fitted_clf).shap_values(X_test_t)
    importance = (pd.DataFrame({"Feature": names, "MeanAbsSHAP": np.abs(shap_values).mean(axis=0)})
                  .sort_values("MeanAbsSHAP", ascending=False))
    print(importance.head(15).to_string(index=False))
    shap.summary_plot(shap_values, X_test_t, feature_names=names, max_display=15)
except ImportError:
    print("shap not installed: pip install shap")

## 11. Customer risk scoring
Bands follow the decision threshold: High = above it, Medium = above half of it, Low = the rest.

In [ ]:
customer_scores = pd.DataFrame({
    "customerID": churn.loc[X_test.index, "customerID"].to_numpy(),
    "ActualChurn": y_test.to_numpy(),
    "ChurnProbability": final_proba,
}, index=X_test.index)

customer_scores["RiskSegment"] = pd.cut(
    customer_scores["ChurnProbability"],
    bins=[-np.inf, final_threshold / 2, final_threshold, np.inf],
    labels=["Low Risk", "Medium Risk", "High Risk"], right=False,
)
actions = {
    "High Risk": "Priority retention outreach",
    "Medium Risk": "Targeted retention offer",
    "Low Risk": "Monitor and maintain engagement",
}
customer_scores["RetentionAction"] = customer_scores["RiskSegment"].map(actions)
customer_scores = customer_scores.sort_values("ChurnProbability", ascending=False)

print(customer_scores.groupby("RiskSegment", observed=True)["ActualChurn"].agg(["size", "mean"]).round(3))
print(customer_scores.head(10).to_string(index=False))

## 12. Save the artifact
The pipeline already contains preprocessing. The threshold, feature list and library version are stored alongside it.

In [ ]:
artifact = {
    "model": final_model,
    "model_name": FINAL_NAME,
    "threshold": final_threshold,
    "feature_columns": X_train.columns.tolist(),
    "sklearn_version": sklearn.__version__,
}
joblib.dump(artifact, "telco_churn_model.joblib")
print("Saved telco_churn_model.joblib")

## Notes
- A churn model predicts risk. It does not show that an intervention reduces churn; validate retention actions with an A/B test.
- The profit curve uses placeholder costs and is only a template for the real analysis.
- This dataset is a public benchmark with no time dimension. A production model needs a time-based split.